# Tests of `TaylorF2_charged.py`

Truncation of the waveform outside $[f_{\rm low}, f_{\rm cut}]$ and deviation of amplitude and
phase from the GR waveform for a negative charge.

In the GR limit, the dephasing due to tidal heating ($q^{\rm BH} = 0$) was also compared with the
code of arXiv:2311.17554 ([gwfit](https://git.ligo.org/samanwaya.mukherjee/gwfit), LIGO GitLab; not distributed here): the two agree to machine precision.

Run from the repository root.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from isco import isco_cutoff
from TaylorF2_charged import TaylorF2_charged

In [ ]:
# Parameters
params = {
    'mass_1': 24.0,
    'mass_2': 8.0,
    'chi_1': 0.7,
    'chi_2': 0.6,
    'distance': 500,
    'inclination': 0.5,
    'phase': 0.0,
    'delta_f': 1.0 / 32.0,
    'f_low': 20.0,
    'f_high': 1024.0
}

In [ ]:
# Tests on truncation of the waveform

hp, _ = TaylorF2_charged(q_BH=0.7, **params)
amp = np.abs(hp)

f_low = params['f_low']
f_high = params['f_high']

# Frequency grid of the same length as hp
freqs = np.arange(len(hp)) * params['delta_f']
q = params['mass_1'] / params['mass_2']
f_cut, chi_f, _ = isco_cutoff(params['mass_1'] + params['mass_2'], q, params['chi_1'], params['chi_2']) # cutoff frequency at 2 * f_isco
print(f"chi_f  = {chi_f:.4f}")
print(f"f_cut  = {f_cut:.2f} Hz")
print(f"f_low  = {f_low} Hz")
print(f"f_high (Nyquist) = {f_high} Hz")

# Check that the mask is working
below_flow  = amp[freqs < f_low]
inside      = amp[(freqs >= f_low) & (freqs <= f_cut)]
above_fcut  = amp[freqs > f_cut]

print(f"\nMax amplitude below f_low:  {np.max(below_flow):.4e}  (expected 0)")
print(f"Max amplitude above f_cut:  {np.max(above_fcut):.4e}  (expected 0)")
print(f"Min/Max amplitude within the range:   {np.min(inside):.4e} / {np.max(inside):.4e}")
print(f"Total number of points:    {len(freqs)}")
print(f"Number of points in [f_low, f_cut]: {np.sum((freqs>=f_low)&(freqs<=f_cut))}")

# Plot
fig, axs = plt.subplots(2, 1, figsize=(10, 8))

# Log-log scale, view of the whole band
axs[0].loglog(freqs[freqs > 0], amp[freqs > 0] + 1e-300)
axs[0].axvline(f_low, color='red',   ls='--', label=f'f_low = {f_low} Hz')
axs[0].axvline(f_cut, color='green', ls='--', label=f'f_cut = {f_cut:.1f} Hz')
axs[0].axvline(f_high, color='gray', ls=':',  label=f'f_high = {f_high} Hz (Nyquist)')
axs[0].set_xlabel("f [Hz]")
axs[0].set_ylabel("|h(f)|")
axs[0].set_title("Amplitude - log-log view of the whole band")
axs[0].legend()
axs[0].grid(alpha=0.3)

# Close view of the cutoff
zoom_mask = (freqs > f_cut - 50) & (freqs < f_cut + 50)
axs[1].plot(freqs[zoom_mask], amp[zoom_mask])
axs[1].axvline(f_cut, color='green', ls='--', label=f'f_cut = {f_cut:.1f} Hz')
axs[1].set_xlabel("f [Hz]")
axs[1].set_ylabel("|h(f)|")
axs[1].set_title("Zoom: transition at f_cut")
axs[1].legend()
axs[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Amplitude and phase when q_BH < 0 (here no horizon for either black hole: complex delta_psi)

# GR waveform (q_BH = 0) and modified waveform
hp_std, _ = TaylorF2_charged(q_BH=0.0, **params)
hp_mod, _ = TaylorF2_charged(q_BH=-0.7, **params)

freqs = np.arange(len(hp_std)) * params['delta_f']
q = params['mass_1'] / params['mass_2']
f_cut, _, _ = isco_cutoff(params['mass_1'] + params['mass_2'], q, params['chi_1'], params['chi_2'])

# Amplitude and phase in [f_low, f_cut]
plot_mask = (freqs >= params['f_low']) & (freqs <= f_cut)

amp_tf2 = np.abs(hp_std[plot_mask])
amp_mod = np.abs(hp_mod[plot_mask])

phase_tf2 = -np.unwrap(np.angle(hp_std[plot_mask])) # unwrap the phase to avoid artificial jumps
phase_mod = -np.unwrap(np.angle(hp_mod[plot_mask]))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

ax1.plot(freqs[plot_mask], amp_tf2, label='GR (q_BH = 0)', color='black', lw=2)
ax1.plot(freqs[plot_mask], amp_mod, label='non-GR (q_BH = -0.7)', color='crimson', linestyle='--')
ax1.set_ylabel("$|\\tilde{h}_+(f)|$")
ax1.set_xlabel("Frequency [Hz]")
ax1.set_title("Amplitude deviation from GR (imaginary part of $\\delta\\psi$)")
ax1.grid(True, linestyle=':')
ax1.legend()

ax2.plot(freqs[plot_mask], phase_mod - phase_tf2, color='darkblue', lw=2, label=r'$\delta \psi$')
ax2.set_xlabel("Frequency [Hz]")
ax2.set_ylabel("$\\Delta\\Phi$ [rad]")
ax2.set_title("Dephasing from GR (real part of $\\delta\\psi$)")
ax2.grid(True, linestyle=':')
ax2.legend()

plt.tight_layout()
plt.show()